# Fine-tune and evaluate on a free Colab T4

Three cells. Select a GPU runtime, then Run All.

1. Cell 1 downloads the runner script and prints the GPU it found. Every line should say `True` and the GPU should be a Tesla T4.
2. Cell 2 fine-tunes Qwen2.5-1.5B-Instruct on 872 verified tool-call traces with LoRA, then scores both the base model and the tuned model on 110 held-out tasks. About 40 minutes.
3. Cell 3 prints the comparison in plain numbers.

What the numbers mean:

- `base accuracy` is how often the untouched model makes the sandbox end in the state the task asked for.
- `tuned accuracy` is the same measurement after fine-tuning.
- `paired delta` compares the two on the same 110 tasks, so it is a per-task win/loss count, not two independent scores.
- `delta on the N items the base already parsed` is the honest one. Those are tasks the base got into valid JSON but the wrong end state, so any movement there is skill rather than learning to format.


In [1]:
import os, pathlib, urllib.request, time, importlib.util

RUNNER = "train_eval_colab.py"
URL = ("https://huggingface.co/datasets/kirankunapuli/cpq-tooluse-verified"
       "/resolve/main/code/train_eval_colab.py")

# Always re-download. A stale copy from an earlier run silently hides fixes.
urllib.request.urlretrieve(URL + f"?t={int(time.time())}", RUNNER)
src = pathlib.Path(RUNNER).read_text()
print("runner bytes:", len(src))
for label, needle in [
    ("T4 logprob fix", "_MM_OUT_DTYPE = False"),
    ("T4 dtype gate", "major >= 8"),
    ("truncation guard", "refusing to train on truncated examples"),
    ("tokenizer batch handling", "def split_batch"),
    ("SFTConfig kwarg filtering", "def make_sft_config"),
]:
    ok = needle in src
    print(f"  {label}: {ok}")
    assert ok, f"downloaded runner is missing {label}; rerun this cell"

import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("compute capability:", torch.cuda.get_device_capability(0))

# Prove the runner picks a dtype this GPU can really use before spending 40 minutes.
spec = importlib.util.spec_from_file_location("runner", RUNNER)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
device, dtype = runner.pick_device(torch)
print(f"will train on: {device} in {dtype}")

if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8:
    assert dtype == torch.float16, (
        f"compute {torch.cuda.get_device_capability(0)} needs fp16, got {dtype}. "
        "The runner on disk is stale, rerun this cell.")


runner bytes: 23108
  T4 logprob fix: True
  T4 dtype fix: True
  tokenizer batch handling: True
  SFTConfig kwarg filtering: True
cuda: True
gpu: Tesla T4
compute capability: (7, 5)
will train on: cuda in torch.bfloat16


In [2]:
import sys, pathlib, importlib.util

SMOKE = False

argv = ["train_eval_colab.py",
        "--base", "Qwen/Qwen2.5-1.5B-Instruct",
        "--epochs", "3", "--lr", "2e-4",
        "--lora-r", "16", "--lora-alpha", "32",
        "--batch", "2", "--grad-accum", "8",
        "--max-length", "2048", "--max-new-tokens", "320",
        "--out", "results.json"]
if SMOKE:
    argv += ["--epochs", "1", "--limit-train", "16", "--limit-test", "8"]

print(" ".join(argv))
print("interpreter:", sys.executable)

# Run inside this kernel instead of shelling out. A subprocess started from sys.executable
# can land on a different interpreter than the one holding torch, which is exactly what
# happened on Colab: the kernel had torch and /usr/bin/python3 did not, so the child died
# before printing anything.
path = pathlib.Path("train_eval_colab.py")
spec = importlib.util.spec_from_file_location("colab_runner", path)
runner = importlib.util.module_from_spec(spec)
sys.modules["colab_runner"] = runner
spec.loader.exec_module(runner)

sys.argv = argv
runner.main()


train_eval_colab.py --base Qwen/Qwen2.5-1.5B-Instruct --epochs 3 --lr 2e-4 --lora-r 16 --lora-alpha 32 --batch 2 --grad-accum 8 --max-length 2048 --max-new-tokens 320 --out results.json


CalledProcessError: Command '['/usr/bin/python3', 'train_eval_colab.py', '--base', 'Qwen/Qwen2.5-1.5B-Instruct', '--epochs', '3', '--lr', '2e-4', '--lora-r', '16', '--lora-alpha', '32', '--batch', '2', '--grad-accum', '8', '--max-length', '2048', '--max-new-tokens', '320', '--out', 'results.json']' returned non-zero exit status 1.

In [ ]:
import json

r = json.load(open("results.json"))
d = r["paired_delta"]
f = r["format"]

print("base model :", r["base_model"])
print("trained on :", r["train_rows"], "examples; tested on", r["test_rows"])
print("max_length :", r["hyperparameters"]["max_length"])
print()
print(f"base accuracy  {r['base']['accuracy']:.3f}", f"({r['base']['correct']}/{r['base']['n']})")
print(f"tuned accuracy {r['tuned']['accuracy']:.3f}", f"({r['tuned']['correct']}/{r['tuned']['n']})")
print(f"paired delta   {d['point']:+.3f}   95% CI [{d['ci'][0]:+.3f}, {d['ci'][1]:+.3f}]")
print()
print("outputs that would not parse:", f["parse_fail_base"], "->", f["parse_fail_tuned"])
print("delta on the", f["n_already_parsed"], "items the base already parsed:",
      f"{f['delta_on_already_parsed']:+.3f}")
